In [164]:
!pip3 install imbalanced-learn


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [165]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

In [166]:
# Loading and preprocessing data
df = pd.read_csv('water_potability.csv')
df = df.fillna(df.mean()) # for any column that has missing values, replace it with that column's mean


In [167]:
# Features/target arrays
X = df.drop('Potability', axis = 1).values  # converting all columns except 'Potability' into NumPy array 
y = df['Potability'].values.reshape(-1, 1)  # target column as a NumPy array, reshaped to a column vector

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42, stratify = y)

In [168]:
# Feature scaling to make sure weights aren't skewed by features with large ranges
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [169]:
# Apply SMOTE on the training set ONLY
smote = SMOTE(random_state = 42)
y_train_1d = y_train.reshape(-1)   
X_train_res, y_train_res = smote.fit_resample(X_train, y_train_1d)

print("Before SMOTE:", X_train.shape, pd.Series(y_train_1d).value_counts().to_dict())
print("After SMOTE:", X_train_res.shape, pd.Series(y_train_res).value_counts().to_dict())

Before SMOTE: (2620, 9) {0: 1598, 1: 1022}
After SMOTE: (3196, 9) {0: 1598, 1: 1598}


In [170]:
# Initializing parameters for logistic regression
m_res, n = X_train_res.shape    # m, n = number of samples and features
w = np.zeros((n, 1))    # weight vector (n, 1) initialized to zeros
b = 0   # scalar bias
alpha = 0.01    # learning rate
epochs = 1000   


In [171]:
# Defining sigmoid function
def sigmoid(z):
    z = np.clip(z, -500, 500)   # to prevent overflow errors
    return 1/(1 + np.exp(-z))

In [172]:
if y_train_res.ndim == 1:
    y_train_res = y_train_res.reshape(-1, 1)
else:
    y_train_res = y_train_res.reshape(-1, 1)  # (m_res,1)


# Training loop: gradient descent
for i in range(epochs):
    z = X_train_res @ w + b
    y_hat = sigmoid(z)
    err = y_hat - y_train_res
    dw = (1/m_res) * (X_train_res.T @ err)
    db = (1/m_res) * np.sum(err)
    w -= alpha * dw
    b -= alpha * db
    if i % 200 == 0:
        cost = -(1/m_res) * np.sum(
            y_train_res*np.log(y_hat+1e-9)+(1-y_train_res)*np.log(1-y_hat+1e-9)
        )
        print(f"epoch {i}: cost={cost:.4f}")
        

epoch 0: cost=0.6931
epoch 200: cost=0.6927
epoch 400: cost=0.6925
epoch 600: cost=0.6925
epoch 800: cost=0.6924


In [173]:
# Evaluation on test set
y_prob = sigmoid(X_test @ w + b).ravel()
y_pred = (y_prob >= 0.5).astype(int)

In [174]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("accuracy:", accuracy_score(y_test, y_pred))

[[215 185]
 [122 134]]
              precision    recall  f1-score   support

           0      0.638     0.537     0.583       400
           1      0.420     0.523     0.466       256

    accuracy                          0.532       656
   macro avg      0.529     0.530     0.525       656
weighted avg      0.553     0.532     0.538       656

accuracy: 0.5320121951219512
